In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("Loading dataset...")
df = pd.read_csv('housing.csv')
print("Initial shape:", df.shape)

# 1. Drop rows with missing values in essential columns first
df = df.dropna(subset=['total_sqft', 'bath', 'price', 'size', 'location'])

# 2. Clean 'total_sqft' (handle range values like '2100-2850')
def convert_sqft_to_num(x):
    try:
        tokens = str(x).split('-')
        if len(tokens) == 2:
            return (float(tokens[0]) + float(tokens[1])) / 2
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna(subset=['total_sqft'])

# 3. Extract BHK from 'size' column (e.g., "2 BHK" -> 2)
df['bhk'] = df['size'].apply(lambda x: int(str(x).split(' ')[0]))

# 4. Handle Locations using One-Hot Encoding
df = pd.get_dummies(df, columns=['location'], drop_first=True)

# 5. Define Features (X) and Target (y) - Explicitly drop columns with NaNs like 'balcony', 'society', etc.
drop_columns = ['price', 'size', 'area_type', 'availability', 'society', 'balcony']
X = df.drop(columns=drop_columns, errors='ignore')
y = df['price']

# Double check: Drop any remaining rows with NaN in X or y to prevent errors
combined = X.copy()
combined['price'] = y
combined = combined.dropna()

X = combined.drop(columns=['price'])
y = combined['price']

# Save the exact column structure so the web app matches later
model_columns = X.columns.tolist()
joblib.dump(model_columns, 'model_columns.pkl')

# 6. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Train Linear Regression Model
print("Training model...")
model = LinearRegression()
model.fit(X_train, y_train)

# Check new score
score = r2_score(y_test, model.predict(X_test)) * 100
print(f"🚀 Improved R-squared Score: {score:.2f}%")

# 8. Save the trained model using Joblib
joblib.dump(model, 'bangalore_house_model.pkl')
print("Model successfully saved as 'bangalore_house_model.pkl'!")

Loading dataset...
Initial shape: (13320, 9)
Training model...
🚀 Improved R-squared Score: 57.99%
Model successfully saved as 'bangalore_house_model.pkl'!
